<a href="https://colab.research.google.com/github/christinrenni123987-debug/data-analysis-toolkit/blob/main/Data_Analysis_Toolkit_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Analysis Toolkit - Colab Version

**Student:** Christin Renni Philip  
**Student ID:** 100008781  
**Course:** Tools and Methods of Data Analysis  
**Sample Dataset:** Netflix Movies and TV Shows

Run each cell from top to bottom.

In [ ]:
# ==========================================================
# SECTION 1: IMPORT REQUIRED LIBRARIES
# Purpose:
# Import all Python libraries needed for data analysis,
# visualization, probability distributions, and statistics.
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

print("Libraries imported successfully.")


Libraries imported successfully.


In [ ]:
# ==========================================================
# SECTION 2: IMPORT DATASET
# Purpose:
# Upload and load the sample CSV dataset into Google Colab.
# When asked, upload the file named: netflix_titles.csv
# ==========================================================

from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename, encoding="latin1")
df.head()


In [ ]:
# ==========================================================
# SECTION 3: DATA EXPLORATION
# Purpose:
# Explore the structure of the dataset including rows,
# columns, column names, data types, and general information.
# ==========================================================

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes)

print("\nDataset Information:")
df.info()


In [ ]:
# ==========================================================
# SECTION 4: MISSING VALUE ANALYSIS
# Purpose:
# Identify missing values and duplicate records.
# This helps understand the quality of the dataset.
# ==========================================================

missing_table = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})

display(missing_table)

print("Duplicate rows:", df.duplicated().sum())


In [ ]:
# ==========================================================
# SECTION 5: DATA CLEANING AND PREPROCESSING
# Purpose:
# Remove duplicate rows and fill missing values.
# Numeric columns are filled using median.
# Categorical columns are filled using mode.
# ==========================================================

df_clean = df.copy()
df_clean = df_clean.drop_duplicates()

numeric_cols = df_clean.select_dtypes(include=np.number).columns
categorical_cols = df_clean.select_dtypes(exclude=np.number).columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    if not df_clean[col].mode().empty:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Data cleaning completed.")
print("Remaining missing values:", df_clean.isnull().sum().sum())

df_clean.head()


In [ ]:
# ==========================================================
# SECTION 6: SELECT NUMERIC COLUMN
# Purpose:
# Select one numeric column for descriptive statistics,
# probability distribution analysis, and inference.
# For this Netflix dataset, use: release_year
# ==========================================================

numeric_columns = df_clean.select_dtypes(include=np.number).columns.tolist()

print("Available numeric columns:")
print(numeric_columns)

column_name = input("Enter numeric column name to analyze, example release_year: ")

data = df_clean[column_name].dropna()

print("Selected column:", column_name)
print("Number of values:", len(data))


In [ ]:
# ==========================================================
# SECTION 7: DESCRIPTIVE STATISTICS
# Purpose:
# Calculate summary measures such as mean, median,
# mode, standard deviation, variance, quartiles,
# skewness, and kurtosis.
# ==========================================================

summary_table = pd.DataFrame({
    "Statistic": [
        "Count", "Mean", "Median", "Mode",
        "Standard Deviation", "Variance",
        "Minimum", "Maximum", "Q1", "Q3",
        "Skewness", "Kurtosis"
    ],
    "Value": [
        data.count(),
        data.mean(),
        data.median(),
        data.mode()[0],
        data.std(),
        data.var(),
        data.min(),
        data.max(),
        data.quantile(0.25),
        data.quantile(0.75),
        data.skew(),
        data.kurtosis()
    ]
})

display(summary_table)


In [ ]:
# ==========================================================
# SECTION 8: DATA VISUALIZATION
# Purpose:
# Create histogram and boxplot to understand distribution,
# spread, central tendency, and possible outliers.
# ==========================================================

plt.figure(figsize=(8, 5))
plt.hist(data, bins=20, edgecolor="black", density=True)
plt.title("Histogram of " + column_name)
plt.xlabel(column_name)
plt.ylabel("Density")
plt.show()

plt.figure(figsize=(8, 4))
plt.boxplot(data, vert=False)
plt.title("Boxplot of " + column_name)
plt.xlabel(column_name)
plt.show()


In [ ]:
# ==========================================================
# SECTION 9: PROBABILITY DISTRIBUTION ANALYSIS
# Purpose:
# Compare the actual data with a fitted normal distribution
# using PDF and CDF plots.
# ==========================================================

mean = data.mean()
std = data.std()

x = np.linspace(data.min(), data.max(), 100)
normal_pdf = stats.norm.pdf(x, mean, std)

plt.figure(figsize=(8, 5))
plt.hist(data, bins=20, density=True, alpha=0.6, edgecolor="black", label="Actual Data")
plt.plot(x, normal_pdf, linewidth=2, label="Fitted Normal Distribution")
plt.title("PDF Plot with Fitted Normal Distribution")
plt.xlabel(column_name)
plt.ylabel("Density")
plt.legend()
plt.show()

sorted_data = np.sort(data)
cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)

plt.figure(figsize=(8, 5))
plt.plot(sorted_data, cdf)
plt.title("CDF Plot of " + column_name)
plt.xlabel(column_name)
plt.ylabel("Cumulative Probability")
plt.show()


In [ ]:
# ==========================================================
# SECTION 10: Q-Q PLOT ANALYSIS
# Purpose:
# Visually check whether the selected numeric data follows
# a normal distribution.
# ==========================================================

plt.figure(figsize=(6, 6))
stats.probplot(data, dist="norm", plot=plt)
plt.title("Q-Q Plot")
plt.show()


In [ ]:
# ==========================================================
# SECTION 11: NORMALITY TESTING
# Purpose:
# Perform Shapiro-Wilk, Kolmogorov-Smirnov, and
# Anderson-Darling tests.
#
# H0: Data follows a normal distribution.
# H1: Data does not follow a normal distribution.
# ==========================================================

alpha = 0.05

# Shapiro-Wilk can warn for very large samples, but it is still useful for demonstration.
shapiro = stats.shapiro(data)
ks = stats.kstest(data, "norm", args=(data.mean(), data.std()))
anderson = stats.anderson(data, dist="norm")

normality_table = pd.DataFrame({
    "Test": ["Shapiro-Wilk", "Kolmogorov-Smirnov", "Anderson-Darling"],
    "Statistic": [shapiro.statistic, ks.statistic, anderson.statistic],
    "p-value / Info": [shapiro.pvalue, ks.pvalue, "Compare statistic with critical value"],
    "Null Hypothesis": ["Data is normally distributed"] * 3,
    "Alternative Hypothesis": ["Data is not normally distributed"] * 3
})

display(normality_table)

print("Shapiro-Wilk decision:")
print("Fail to reject H0" if shapiro.pvalue > alpha else "Reject H0")

print("\nKolmogorov-Smirnov decision:")
print("Fail to reject H0" if ks.pvalue > alpha else "Reject H0")

print("\nAnderson-Darling details:")
print("Statistic:", anderson.statistic)
print("Critical values:", anderson.critical_values)
print("Significance levels:", anderson.significance_level)


In [ ]:
# ==========================================================
# SECTION 12: CONFIDENCE INTERVAL ESTIMATION
# Purpose:
# Calculate a 95% confidence interval for the population
# mean of the selected numeric column.
# ==========================================================

confidence = 0.95

sample_mean = data.mean()
standard_error = stats.sem(data)

ci = stats.t.interval(
    confidence,
    len(data) - 1,
    loc=sample_mean,
    scale=standard_error
)

print("Sample mean:", sample_mean)
print("95% Confidence Interval:", ci)


In [ ]:
# ==========================================================
# SECTION 13: HYPOTHESIS TESTING
# Purpose:
# Perform a one-sample t-test to compare the sample mean
# against a hypothesized value.
#
# H0: Sample mean equals the hypothesized mean.
# H1: Sample mean is different from the hypothesized mean.
# ==========================================================

hypothesized_mean = float(input("Enter hypothesized mean value, example 2015: "))

t_result = stats.ttest_1samp(data, hypothesized_mean)

print("H0: Mean =", hypothesized_mean)
print("H1: Mean !=", hypothesized_mean)
print("T-statistic:", t_result.statistic)
print("p-value:", t_result.pvalue)

if t_result.pvalue > alpha:
    print("Decision: Fail to reject H0")
    print("Conclusion: The sample mean is not significantly different from the hypothesized mean.")
else:
    print("Decision: Reject H0")
    print("Conclusion: The sample mean is significantly different from the hypothesized mean.")


In [ ]:
# ==========================================================
# SECTION 14: CORRELATION ANALYSIS
# Purpose:
# Create a correlation heatmap to check relationships
# between numeric variables.
# ==========================================================

numeric_df = df_clean.select_dtypes(include=np.number)

if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr()

    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Correlation Heatmap")
    plt.show()
else:
    print("This dataset does not have enough numeric columns for correlation analysis.")


In [ ]:
# ==========================================================
# SECTION 15: EXPORT CLEANED DATASET
# Purpose:
# Save the cleaned dataset so it can be reused,
# downloaded, or submitted with the analysis.
# ==========================================================

df_clean.to_csv("cleaned_netflix_dataset.csv", index=False)

print("Cleaned dataset exported as cleaned_netflix_dataset.csv")


## Final Toolkit Conclusion

Based on the visual inspection of the histogram, PDF plot, and Q–Q plot, along with the results of the Shapiro–Wilk, Kolmogorov–Smirnov, and Anderson–Darling normality tests, the distribution of the selected variable was evaluated for normality. If the p-values obtained from the Shapiro–Wilk and Kolmogorov–Smirnov tests are greater than the significance level of 0.05, we fail to reject the null hypothesis that the data are normally distributed. Similarly, if the Anderson–Darling test statistic is lower than the corresponding critical value, the results further support the assumption of normality. Therefore, the data can be considered approximately normally distributed, indicating that the assumption of normality is reasonably satisfied and that parametric statistical methods can be appropriately applied to this dataset.
